# VRT Data Preparation

> Building a recall dataset from human-scored film memory reports.

In the Vigilance Reaction Task (VRT), participants watch a 12-minute film composed of 11 short clips — either emotionally distressing or neutral — and later perform a computerised task where film-related images occasionally appear on screen. When a clip comes to mind, the participant presses the spacebar and types what they remember.

The raw data has two layers. The **Excel exports** record every event in the task row by row: digits shown, background images, button presses, and typed text. The **scored documents** are Word files where human raters read each typed response and labelled which of the 11 clips it refers to, using tags like `[clip_three]` or `[invalid_press]`.

Each participant completed two VRT sessions — one measuring involuntary (intrusion) recall and one measuring voluntary (free) recall — but the scored documents only cover the first session. This notebook therefore produces a dataset of 240 single-session trials rather than the full 480.

We combine both data layers into a single HDF5 dataset suitable for recall analyses (serial position curves, conditional response probability, etc.), proceeding in stages:

1. Parse the scored documents to get human-coded clip assignments
2. Parse the raw Excel files for event-stream data (cue images, timing)
3. Link the two via subject identifiers
4. Assemble the integer arrays and export

## Table of contents

- [Setup](#setup)
- [Parsing scored documents](#parsing-scored-documents)
  - [Reading a scored document](#reading-a-scored-document)
  - [Parsing button presses](#parsing-button-presses)
  - [Processing all scored files](#processing-all-scored-files)
    - [Summary statistics](#summary-statistics)
    - [Multi-clip presses](#multiclip-presses)
- [Parsing raw Excel exports](#parsing-raw-excel-exports)
  - [Reading an Excel file](#reading-an-excel-file)
  - [Parsing cue and foil codes](#parsing-cue-and-foil-codes)
  - [Processing all Excel files](#processing-all-excel-files)
    - [Summary statistics](#summary-statistics)
- [Linking scored documents to Excel files](#linking-scored-documents-to-excel-files)
- [Loading participant metadata](#loading-participant-metadata)
- [Building output sequences](#building-output-sequences)
  - [Validating press alignment](#validating-press-alignment)
  - [Building the sequences](#building-the-sequences)
- [Assembling HDF5 arrays and exporting](#assembling-hdf5-arrays-and-exporting)
  - [QC summary](#qc-summary)
  - [Spot-checking against scored documents](#spotchecking-against-scored-documents)

## Setup

In [1]:
import re
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from pathlib import Path

from jaxcmr.helpers import find_project_root

In [2]:
ROOT = Path(find_project_root())
SCORED_DIR = ROOT / "data" / "raw" / "Scored_VRT data files"
assert SCORED_DIR.is_dir(), f"Not found: {SCORED_DIR}"

## Parsing scored documents

### Reading a scored document

The scored files are `.docx` Word documents produced by [Taguette](https://www.taguette.org), a qualitative coding tool. Each file covers one participant's first VRT session. The filename encodes the rater and a 5-letter participant code, e.g. `Tommy_AGMWB_1st.docx`. Tommy scored all 240 participants; we use his files exclusively.

Inside, the text is a flat sequence of tagged recall reports. A `[button_press]` marker starts each report, followed by a clip label and the participant's typed text with interspersed detail tags:

```
[button_press] [clip_three] I remember kids playing football [Internal_tag]
in a garden [Internal_tag] and an accident happened [Internal_tag]
```

A `.docx` file is just a zip archive containing XML. We extract the paragraph text directly — no need for the `python-docx` library.

In [3]:
def read_docx_paragraphs(path: Path) -> list[str]:
    """Return the text of each paragraph in a .docx file."""
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    with zipfile.ZipFile(path) as z:
        tree = ET.parse(z.open("word/document.xml"))
    paragraphs = []
    for p in tree.findall(".//w:p", ns):
        texts = [t.text or "" for t in p.findall(".//w:t", ns)]
        paragraphs.append("".join(texts))
    return paragraphs

Let's read one file and look at what we get:

In [4]:
sample_path = SCORED_DIR / "Tommy_AGMWB_1st.docx"
sample_paragraphs = read_docx_paragraphs(sample_path)

for i, p in enumerate(sample_paragraphs):
    if p.strip():
        print(f"[{i}] {p}")

[0] Tommy_AGMWB_1st
[1] [button_press] [clip_nine] I remember someone performing a surgery on the skin of a person [Internal_tag]
[3] [button_press] [clip_six] I rememeber the genocide in Rwanda [Internal_tag] where 800.000 people were killed [Internal_tag]
[5] [button_press] [clip_three] I remember kids playing football [Internal_tag] in a garden [Internal_tag] and an accident happened with cars [Internal_tag] and one car killed one of the kids [Internal_tag]
[7] [button_press] [clip_eleven] An elephant attacking an injuring a person [Internal_tag] and the going outside the circus [Internal_tag] and attacking more people [Internal_tag]
[9] [button_press] [clip_eight] a couple in love sitting on a wall [Internal_tag] and a car crash happened [Internal_tag] and one of the cars killed the guy [Internal_tag]
[11] [button_press] [clip_four] someone with atool was trying to clean an eye [Internal_tag]
[13] [button_press] [clip_five] a couple was inside the car [Internal_tag] very happy [Int

Each non-empty paragraph is one button-press event. The first paragraph is just the filename header. Everything else follows the pattern:

```
[button_press] [clip_LABEL] <utterance text with [tag] markers>
```

### Parsing button presses

From each button-press block we extract three things:

1. **Clip labels** — which of the 11 clips the rater assigned (`clip_one` .. `clip_eleven`, `clip_unknown`, or `invalid_press`). These become the `recalls` array in the final HDF5.
2. **Order** — the sequence in which clips were recalled, which is what CRP and serial-position analyses operate on.
3. **Utterance text** — what the participant typed. We'll need this later to match 5-letter subject codes back to the numeric IDs in the raw Excel files, since the text appears in both sources.

Occasional multi-clip assignments (e.g., `[clip_one, clip_two]`) mean we can't just grab the first clip label per block. Instead we:

1. Join all paragraphs into one string (tags sometimes spill across paragraph breaks)
2. Split on `[button_press` to get one chunk per press
3. Within each chunk, find all clip labels and strip tag markers

In [5]:
CLIP_WORD_TO_NUMBER = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10,
    "eleven": 11,
}


@dataclass
class ButtonPress:
    """One scored button-press event from a VRT session."""

    clip_numbers: list[int]  # 1-11 for identified clips; empty for invalid
    clip_unknown: bool  # True if rater could not identify the clip
    invalid: bool  # True if rater marked as invalid_press
    text: str  # Participant's typed response, tags stripped


# Patterns for clip labels and detail tags.
_CLIP_RE = re.compile(r"clip_(\w+)")
_TAG_RE = re.compile(
    r"\[(Internal_tag|External_tag|Repetition_tag|Error_tag|Other_tag)\]"
)
_BRACKET_RE = re.compile(r"\[[^\[\]]*\]")


def parse_button_presses(paragraphs: list[str]) -> list[ButtonPress]:
    """Parse all button-press events from a scored document's paragraphs."""
    # Join paragraphs with spaces — tags sometimes wrap across lines.
    full_text = " ".join(p.strip() for p in paragraphs)

    # Split on [button_press to isolate each event.
    # The first chunk is the header (filename), which we discard.
    chunks = re.split(r"\[button_press", full_text)
    if len(chunks) < 2:
        return []

    presses = []
    for chunk in chunks[1:]:
        invalid = "invalid_press" in chunk

        # Find all clip_X labels in this chunk.
        clip_numbers = []
        clip_unknown = False
        for match in _CLIP_RE.finditer(chunk):
            word = match.group(1)
            if word == "unknown":
                clip_unknown = True
            elif word in CLIP_WORD_TO_NUMBER:
                clip_numbers.append(CLIP_WORD_TO_NUMBER[word])

        # Strip all bracket markers to recover the plain utterance text.
        text = _BRACKET_RE.sub("", chunk)
        # Also strip the leading "] " or "] " left from the button_press bracket.
        text = re.sub(r"^[\]\s]+", "", text).strip()
        # Collapse whitespace.
        text = re.sub(r"\s+", " ", text)

        presses.append(
            ButtonPress(
                clip_numbers=clip_numbers,
                clip_unknown=clip_unknown,
                invalid=invalid,
                text=text,
            )
        )

    return presses

Let's try it on our sample file:

In [6]:
sample_presses = parse_button_presses(sample_paragraphs)

for i, bp in enumerate(sample_presses, 1):
    label = ", ".join(str(c) for c in bp.clip_numbers) or (
        "unknown" if bp.clip_unknown else "invalid"
    )
    print(f"  {i:2d}. clip {label:>7s}  {bp.text[:80]}")

print(
    f"\n{len(sample_presses)} button presses, "
    f"{sum(1 for bp in sample_presses if bp.clip_numbers)} with clip labels"
)

   1. clip       9  I remember someone performing a surgery on the skin of a person
   2. clip       6  I rememeber the genocide in Rwanda where 800.000 people were killed
   3. clip       3  I remember kids playing football in a garden and an accident happened with cars 
   4. clip      11  An elephant attacking an injuring a person and the going outside the circus and 
   5. clip       8  a couple in love sitting on a wall and a car crash happened and one of the cars 
   6. clip       4  someone with atool was trying to clean an eye
   7. clip       5  a couple was inside the car very happy sitting on the back sit and then their ca
   8. clip       7  someone was in the sea sea trying to survive but he couldnt and I think he died
   9. clip       2  someone was shaving and while he was shaving he was bleeding

9 button presses, 9 with clip labels


That file had no multi-clip presses. Here's one that does — participant BQFXW has a button press tagged with two clips at once:

```
[button_press [clip_three, clip_eight]] drunk driving advert [Internal_tag]
moments before man gets crushed by car [Internal_tag, Error_tag]
```

The participant recalled content spanning clips 3 and 8 in a single response. Because `_CLIP_RE` finds *all* `clip_X` matches in the chunk, both end up in `clip_numbers`:

In [7]:
multi_paragraphs = read_docx_paragraphs(SCORED_DIR / "Tommy_BQFXW_1st.docx")
multi_presses = parse_button_presses(multi_paragraphs)

for i, bp in enumerate(multi_presses, 1):
    if len(bp.clip_numbers) > 1:
        print(f"Press {i}: clip_numbers={bp.clip_numbers}")
        print(f"  text: {bp.text}")
        break

Press 10: clip_numbers=[3, 8]
  text: drunk driving advert moments before man gets crushed by car


### Processing all scored files

Tommy scored all 240 participants. We glob his files and parse each one.

In [8]:
def subject_code_from_path(path: Path) -> str:
    """Return the 5-letter subject code from a scored .docx filename."""
    # e.g. Tommy_AGMWB_1st.docx -> "AGMWB"
    return path.stem.split("_")[1]

In [9]:
all_files = sorted(SCORED_DIR.glob("Tommy_*_1st.docx"))
print(f"{len(all_files)} scored files")

240 scored files


In [10]:
@dataclass
class ScoredSession:
    """Parsed data from one scored VRT document."""

    subject_code: str  # 5-letter participant code
    presses: list[ButtonPress]

    @property
    def recall_clips(self) -> list[int]:
        """Clip numbers in button-press order, including repeats."""
        clips = []
        for bp in self.presses:
            clips.extend(bp.clip_numbers)
        return clips

    @property
    def unique_clips(self) -> list[int]:
        """Clip numbers in first-mention order, no repeats."""
        seen = set()
        clips = []
        for c in self.recall_clips:
            if c not in seen:
                seen.add(c)
                clips.append(c)
        return clips

In [11]:
scored_sessions: dict[str, ScoredSession] = {}

for path in all_files:
    code = subject_code_from_path(path)
    paragraphs = read_docx_paragraphs(path)
    scored_sessions[code] = ScoredSession(
        subject_code=code,
        presses=parse_button_presses(paragraphs),
    )

print(f"{len(scored_sessions)} sessions")

240 sessions


#### Summary statistics

In [12]:
import numpy as np

n_presses = [len(s.presses) for s in scored_sessions.values()]
n_valid = [
    sum(1 for bp in s.presses if bp.clip_numbers) for s in scored_sessions.values()
]
n_invalid = [sum(1 for bp in s.presses if bp.invalid) for s in scored_sessions.values()]
n_unknown = [
    sum(1 for bp in s.presses if bp.clip_unknown) for s in scored_sessions.values()
]
n_unique = [len(s.unique_clips) for s in scored_sessions.values()]
n_raw = [len(s.recall_clips) for s in scored_sessions.values()]


def describe(label, values):
    a = np.array(values)
    print(
        f"  {label:30s}  mean={a.mean():.1f}  median={np.median(a):.0f}  "
        f"min={a.min()}  max={a.max()}"
    )


print(f"{len(scored_sessions)} sessions\n")
describe("Button presses per session", n_presses)
describe("Valid (has clip label)", n_valid)
describe("Invalid presses", n_invalid)
describe("Clip unknown", n_unknown)
describe("Unique clips recalled", n_unique)
describe("Total clip mentions (raw)", n_raw)

240 sessions

  Button presses per session      mean=17.6  median=16  min=0  max=75
  Valid (has clip label)          mean=16.5  median=15  min=0  max=74
  Invalid presses                 mean=0.7  median=0  min=0  max=22
  Clip unknown                    mean=0.4  median=0  min=0  max=9
  Unique clips recalled           mean=8.4  median=9  min=0  max=11
  Total clip mentions (raw)       mean=16.7  median=15  min=0  max=74


In [13]:
# Filter out empty sessions (zero button presses) so downstream steps don't need to handle them.
empty_codes = sorted(code for code, s in scored_sessions.items() if not s.presses)
for code in empty_codes:
    del scored_sessions[code]

print(f"Removed {len(empty_codes)} empty sessions: {empty_codes}")
print(f"{len(scored_sessions)} sessions remaining")

Removed 3 empty sessions: ['RQCUW', 'RQPAU', 'WQTSF']
237 sessions remaining


#### Multi-clip presses

Some button presses are tagged with more than one clip (the participant recalled content from multiple clips in a single response). Let's see how common this is and what they look like:

In [14]:
multi_clip_count = 0
multi_clip_examples = []

for s in scored_sessions.values():
    for bp in s.presses:
        if len(bp.clip_numbers) > 1:
            multi_clip_count += 1
            if len(multi_clip_examples) < 5:
                multi_clip_examples.append((s.subject_code, bp))

total_presses = sum(len(s.presses) for s in scored_sessions.values())
print(
    f"{multi_clip_count} multi-clip presses out of {total_presses} total "
    f"({100 * multi_clip_count / total_presses:.1f}%)\n"
)

for code, bp in multi_clip_examples:
    print(f"  {code}: clips {bp.clip_numbers}")
    print(f"    {bp.text[:120]}\n")

29 multi-clip presses out of 4215 total (0.7%)

  BQFXW: clips [3, 8]
    drunk driving advert moments before man gets crushed by car

  CHHIO: clips [1, 2]
    Once again the woman teaching . girl throwing the beanbag, rethrowingit. I've also just remember that the guy in the caf

  FIPFF: clips [5, 2, 1]
    the man driving has a british accent and the teacher had an american accent , I noticed a number of accents in the coffe

  IDJRT: clips [8, 5]
    The other scene is about a man show the steps how to ride a forklift , he show the how to run it and make a turn . Anoth

  IDJRT: clips [11, 7]
    There are two scenes about the sport. The first one is two woman standing in the first then a womanshow the tutorial of 



## Parsing raw Excel exports

The scored documents tell us *which* clips each participant recalled, but not the surrounding task context: which film-related cue images appeared on screen before each button press, what foil images were shown, or the full row-by-row event stream. That information lives in the raw Excel exports at `data/raw/VRT_data_unprocessed/`.

Each participant has two `.xlsx` files — one per task — named like `p100_Intrusion_Original.xlsx` and `p100_FreeRecall_Original.xlsx`. Each file has one worksheet with ~270 rows covering the full VRT session. The columns are:

| Column | Content |
|--------|---------|
| A | Trial number (sequential) |
| B | Digit shown on screen (1–9) |
| C | Background code: `"black"`, `"scene1"`..`"scene68"` (foils), or `"7_1"` etc. (film cues) |
| D | Background type: 1 = film cue, 2 = foil scene, 3 = black |
| E | Timestamp (ms) |
| F | Response: 1 = digit pressed, 2 = withheld, 3 = spacebar (memory report) |
| H | Free text typed by participant (only when F = 3) |

Like the scored `.docx` files, `.xlsx` files are zip archives of XML. We parse them the same way — no `openpyxl` or `pandas` needed.

### Reading an Excel file

In [15]:
EXCEL_DIR = ROOT / "data" / "raw" / "VRT_data_unprocessed"
assert EXCEL_DIR.is_dir(), f"Not found: {EXCEL_DIR}"

In [16]:
_SHEET_NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}


def _xlsx_cell_to_value(cell, shared_strings: list[str]) -> str | None:
    """Decode a single cell element from an xlsx worksheet."""
    value = cell.find("m:v", _SHEET_NS)
    if value is None or value.text is None:
        return None
    if cell.attrib.get("t") == "s":
        return shared_strings[int(value.text)]
    return value.text


def read_xlsx_rows(path: Path) -> list[tuple[int, dict[str, str | None]]]:
    """Return rows from the first worksheet as (row_index, {column: value}) pairs."""
    with zipfile.ZipFile(path) as wb:
        shared_strings = [
            t.text or ""
            for t in ET.fromstring(wb.read("xl/sharedStrings.xml")).iter(
                "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t"
            )
        ]
        sheet = ET.fromstring(wb.read("xl/worksheets/sheet1.xml"))

    rows = []
    for row in sheet.findall("m:sheetData/m:row", _SHEET_NS):
        row_index = int(row.attrib["r"])
        values: dict[str, str | None] = {}
        for cell in row.findall("m:c", _SHEET_NS):
            ref = cell.attrib.get("r", "")
            col_match = re.match(r"[A-Z]+", ref)
            if col_match:
                values[col_match.group(0)] = _xlsx_cell_to_value(cell, shared_strings)
        rows.append((row_index, values))
    return rows

In [17]:
@dataclass
class VrtRow:
    """One row from a VRT Excel export."""

    row_index: int
    trial: int | None  # A: trial number
    digit: int | None  # B: digit shown (1-9)
    bg_code: str | None  # C: background code ("black", "scene1", "7_1", ...)
    bg_type: int | None  # D: 1=film cue, 2=foil, 3=black
    response: int | None  # F: 1=digit press, 2=withheld, 3=spacebar
    text: str | None  # H: free text (only when response=3)


def read_vrt_xlsx(path: Path) -> list[VrtRow]:
    """Read a VRT Excel export into typed rows."""

    def to_int(v: str | None) -> int | None:
        return int(v) if v is not None and v != "" else None

    return [
        VrtRow(
            row_index=ri,
            trial=to_int(vals.get("A")),
            digit=to_int(vals.get("B")),
            bg_code=vals.get("C"),
            bg_type=to_int(vals.get("D")),
            response=to_int(vals.get("F")),
            text=vals.get("H"),
        )
        for ri, vals in read_xlsx_rows(path)
    ]

Let's read one file and look at the rows around a memory report (response = 3). Most rows are digit-task events on a black background; the interesting ones are film cues (D=1) and spacebar presses with text (F=3):

In [18]:
sample_xlsx = EXCEL_DIR / "p100_Intrusion_Original.xlsx"
sample_rows = read_vrt_xlsx(sample_xlsx)

# Find the first spacebar press and show a window around it.
for i, row in enumerate(sample_rows):
    if row.response == 3:
        window = sample_rows[max(0, i - 3) : i + 3]
        for r in window:
            marker = " <-- cue" if r.bg_type == 1 else (" <-- text" if r.text else "")
            print(
                f"  row {r.row_index:3d}  D={r.bg_type}  F={r.response}  "
                f"C={str(r.bg_code):>10s}  H={r.text or ''}{marker}"
            )
        break

print(
    f"\n{len(sample_rows)} rows total, "
    f"{sum(1 for r in sample_rows if r.bg_type == 1)} cue events, "
    f"{sum(1 for r in sample_rows if r.text)} text entries"
)

  row  10  D=3  F=1  C=     black  H=
  row  11  D=3  F=1  C=     black  H=
  row  12  D=1  F=1  C=       7_1  H= <-- cue
  row  13  D=3  F=3  C=     black  H=2 instructors at the gym  <-- text
  row  14  D=3  F=None  C=     black  H=
  row  15  D=2  F=1  C=    scene4  H=

270 rows total, 22 cue events, 11 text entries


### Parsing cue and foil codes

Column C encodes what's on screen. When the background is a film cue (D=1), column C contains a code like `"7_1"` or `"7_IPT"` where the leading integer is the clip number (1–11). When it's a foil (D=2), column C is `"scene1"` through `"scene68"`. We parse these into integers:

In [19]:
def parse_cue_clip(code: str | None) -> int | None:
    """Parse a film cue clip number (1-11) from a column C code like '7_1'."""
    if code is None:
        return None
    m = re.match(r"^(\d+)", code.strip())
    if m is None:
        return None
    n = int(m.group(1))
    return n if 1 <= n <= 11 else None


def parse_foil_scene(code: str | None) -> int | None:
    """Parse a foil scene number (1-68) from a column C code like 'scene5'."""
    if code is None:
        return None
    m = re.match(r"^scene(\d+)$", code.strip(), re.IGNORECASE)
    if m is None:
        return None
    n = int(m.group(1))
    return n if 1 <= n <= 68 else None

In [20]:
# Show the distinct cue codes from our sample file.
cue_rows = [r for r in sample_rows if r.bg_type == 1]
foil_rows = [r for r in sample_rows if r.bg_type == 2]

print("Film cue codes and parsed clip numbers:")
for r in cue_rows[:6]:
    print(f"  C={r.bg_code!r:>10s}  -> clip {parse_cue_clip(r.bg_code)}")
print(f"  ... ({len(cue_rows)} cue events total)\n")

print("Foil codes and parsed scene numbers:")
for r in foil_rows[:4]:
    print(f"  C={r.bg_code!r:>12s}  -> scene {parse_foil_scene(r.bg_code)}")
print(f"  ... ({len(foil_rows)} foil events total)")

Film cue codes and parsed clip numbers:
  C=     '7_1'  -> clip 7
  C=     '1_1'  -> clip 1
  C=     '3_1'  -> clip 3
  C=     '8_1'  -> clip 8
  C=     '5_1'  -> clip 5
  C=     '4_1'  -> clip 4
  ... (22 cue events total)

Foil codes and parsed scene numbers:
  C=    'scene1'  -> scene 1
  C=    'scene2'  -> scene 2
  C=    'scene3'  -> scene 3
  C=    'scene4'  -> scene 4
  ... (68 foil events total)


### Processing all Excel files

Each filename encodes the subject's numeric ID and task type. We parse these, read every file, and store the row streams keyed by `(subject_id, task)`. There are ~480 files because each participant has two sessions (intrusion and free recall), though we only have scored documents for the first.

In [21]:
def parse_subject_and_task(path: Path) -> tuple[int, str]:
    """Return (subject_id, task) from a VRT filename like 'p100_Intrusion_Original.xlsx'."""
    m = re.match(r"^p(\d+)_", path.name)
    if m is None:
        raise ValueError(f"Cannot parse subject from: {path.name}")
    subject = int(m.group(1))
    if "_Intrusion_" in path.name:
        return subject, "intrusion"
    if "_FreeRecall_" in path.name:
        return subject, "free_recall"
    raise ValueError(f"Cannot parse task from: {path.name}")

In [22]:
excel_files = sorted(EXCEL_DIR.glob("p*_*.xlsx"))
print(f"{len(excel_files)} Excel files found")

# Keep one file per (subject, task) — if duplicates exist, take the last alphabetically.
files_by_key: dict[tuple[int, str], Path] = {}
for path in excel_files:
    try:
        key = parse_subject_and_task(path)
    except ValueError:
        continue
    existing = files_by_key.get(key)
    if existing is None or path.name > existing.name:
        files_by_key[key] = path

subjects = sorted({s for s, _ in files_by_key})
print(
    f"{len(files_by_key)} unique (subject, task) pairs across {len(subjects)} subjects"
)

480 Excel files found
480 unique (subject, task) pairs across 240 subjects


Reading all ~480 files takes a moment. We store the parsed rows so later sections can align cues to scored recalls and build the event-stream arrays.

In [23]:
rows_by_key: dict[tuple[int, str], list[VrtRow]] = {}

for key in sorted(files_by_key):
    rows_by_key[key] = read_vrt_xlsx(files_by_key[key])

print(f"{len(rows_by_key)} sessions loaded")

480 sessions loaded


#### Summary statistics

In [24]:
n_rows = [len(rows) for rows in rows_by_key.values()]
n_cues = [sum(1 for r in rows if r.bg_type == 1) for rows in rows_by_key.values()]
n_foils = [sum(1 for r in rows if r.bg_type == 2) for rows in rows_by_key.values()]
n_texts = [sum(1 for r in rows if r.text) for rows in rows_by_key.values()]

print(f"{len(rows_by_key)} sessions\n")
describe("Rows per session", n_rows)
describe("Film cue events", n_cues)
describe("Foil events", n_foils)
describe("Text entries (spacebar)", n_texts)

480 sessions

  Rows per session                mean=270.0  median=270  min=270  max=270
  Film cue events                 mean=22.0  median=22  min=22  max=22
  Foil events                     mean=68.0  median=68  min=68  max=68
  Text entries (spacebar)         mean=17.8  median=16  min=0  max=230


## Linking scored documents to Excel files

The scored documents use 5-letter participant codes (like `AGMWB`), while the raw Excel files use numeric IDs (`p100`). No lookup table maps between them, so we derive the mapping from the typed text itself: column H in the Excel contains the raw participant input, and the scored documents wrap that same text with clip and detail tags. After tag stripping, the texts *almost* match — but tag removal introduces stray spaces before punctuation (e.g., `room . We` instead of `room. We`), and occasional Word XML escape sequences like `_x001b_` leave artifacts.

We normalize both sides aggressively — remove XML escapes, strip all punctuation, lowercase — then build a reverse index from each normalized text to the Excel sessions that contain it. For each scored session we look up its texts and vote on the best-matching Excel session.

In [25]:
def normalize_text(text: str) -> str:
    """Normalize for matching: strip XML escapes, remove punctuation, lowercase."""
    text = re.sub(r"_x[0-9a-fA-F]{4}_", "", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()


# Collect normalized utterance texts from each Excel session.
excel_utterances: dict[tuple[int, str], list[str]] = {}
for key, rows in rows_by_key.items():
    excel_utterances[key] = [
        normalize_text(r.text) for r in rows if r.text and r.text.strip()
    ]

# Reverse index: normalized text -> set of Excel sessions containing it.
text_index: dict[str, set[tuple[int, str]]] = {}
for key, texts in excel_utterances.items():
    for t in texts:
        if t:
            text_index.setdefault(t, set()).add(key)

print(
    f"{len(text_index)} unique utterance texts across {len(rows_by_key)} Excel sessions"
)

7303 unique utterance texts across 480 Excel sessions


In [26]:
# For each scored session, find the Excel session whose utterance texts overlap most.
code_to_key: dict[str, tuple[int, str]] = {}

for code, session in sorted(scored_sessions.items()):
    scored_texts = [
        normalize_text(bp.text) for bp in session.presses if bp.text.strip()
    ]

    votes: dict[tuple[int, str], int] = {}
    for t in scored_texts:
        for key in text_index.get(t, set()):
            votes[key] = votes.get(key, 0) + 1

    if not votes:
        raise ValueError(f"No text matches for {code} — check normalization")

    code_to_key[code] = max(votes, key=lambda k: votes[k])

matched_pids = sorted({pid for pid, _ in code_to_key.values()})
print(
    f"Matched all {len(code_to_key)} scored sessions to {len(matched_pids)} distinct pIDs"
)

Matched all 237 scored sessions to 237 distinct pIDs


In [27]:
# Validate: what fraction of each session's scored texts appear in the matched Excel file?
match_rates = []
for code, key in code_to_key.items():
    scored_texts = [
        normalize_text(bp.text)
        for bp in scored_sessions[code].presses
        if bp.text.strip()
    ]
    excel_set = set(excel_utterances[key])
    n_hit = sum(1 for t in scored_texts if t in excel_set)
    match_rates.append(n_hit / len(scored_texts))

describe("Text match rate (%)", [r * 100 for r in match_rates])

# Confirm 1-to-1 mapping (no Excel session claimed by multiple scored files).
reverse: dict[tuple[int, str], list[str]] = {}
for code, key in code_to_key.items():
    reverse.setdefault(key, []).append(code)

dupes = {k: v for k, v in reverse.items() if len(v) > 1}
if dupes:
    print(f"\nWARNING: {len(dupes)} Excel sessions matched by multiple codes")
    for k, codes in sorted(dupes.items()):
        print(f"  p{k[0]} {k[1]} <- {codes}")
else:
    print(f"All {len(code_to_key)} matches are 1-to-1")

# A few examples.
print("\nSample matches:")
for code in sorted(code_to_key)[:5]:
    pid, task = code_to_key[code]
    n = len([bp for bp in scored_sessions[code].presses if bp.text.strip()])
    print(f"  {code} -> p{pid} ({task}), {n} button presses")

  Text match rate (%)             mean=98.2  median=100  min=66.66666666666666  max=100.0
All 237 matches are 1-to-1

Sample matches:
  AGMWB -> p103 (free_recall), 9 button presses
  AJLYW -> p219 (intrusion), 11 button presses
  ASRNG -> p40 (free_recall), 30 button presses
  AUQGT -> p78 (intrusion), 3 button presses
  AXWOO -> p257 (intrusion), 19 button presses


## Loading participant metadata

The PID tags spreadsheet at `data/raw/VRP_pid_tags.xlsx` records three key variables for each participant:

- **emotion** (0 = neutral film, 1 = emotional film) — determines which set of 11 clips they saw
- **intervention** (0 or 1) — whether the participant received the experimental intervention
- **intentionality** (1 or 2) — the order of VRT sessions. Intentionality 1 means the first session was the *intrusion* (involuntary recall) task; intentionality 2 means it was *free recall* (voluntary). Since our scored documents all cover the first session, this tells us which task each participant's scored data belongs to.

We also use the metadata to resolve the three empty sessions that text matching couldn't identify: whichever pIDs aren't yet in our mapping must correspond to those three participants.

In [28]:
PID_TAGS_PATH = ROOT / "data" / "raw" / "VRP_pid_tags.xlsx"
assert PID_TAGS_PATH.is_file(), f"Not found: {PID_TAGS_PATH}"

# Read the PID tags spreadsheet.
pid_rows = read_xlsx_rows(PID_TAGS_PATH)

# First row is the header; find columns by name.
_, header = pid_rows[0]
col_of = {}
for col, val in header.items():
    if val is not None:
        col_of[val.strip().lower()] = col


def req_int(val: str | None) -> int:
    return int(float(val)) if val else 0


@dataclass
class PidTags:
    subject: int
    emotion: int  # 0=neutral, 1=emotional
    intervention: int  # 0 or 1
    intentionality: int  # 1=intrusion-first, 2=free_recall-first


pid_tags: dict[int, PidTags] = {}
for _, vals in pid_rows[1:]:
    pid_raw = vals.get(col_of["pid"])
    if pid_raw is None or pid_raw.strip() == "":
        continue
    pid = int(float(pid_raw))
    pid_tags[pid] = PidTags(
        subject=pid,
        emotion=req_int(vals.get(col_of["emotion"])),
        intervention=req_int(vals.get(col_of["intervention"])),
        intentionality=req_int(vals.get(col_of["intentionality"])),
    )

print(f"{len(pid_tags)} participants in PID tags")
print(
    f"  emotion: {sum(1 for t in pid_tags.values() if t.emotion == 0)} neutral, "
    f"{sum(1 for t in pid_tags.values() if t.emotion == 1)} emotional"
)
print(
    f"  intentionality: {sum(1 for t in pid_tags.values() if t.intentionality == 1)} intrusion-first, "
    f"{sum(1 for t in pid_tags.values() if t.intentionality == 2)} free_recall-first"
)

480 participants in PID tags
  emotion: 240 neutral, 240 emotional
  intentionality: 240 intrusion-first, 240 free_recall-first


In [29]:
# The 3 empty sessions we filtered earlier are the 3 pIDs not yet in our mapping.
matched_pids = {key[0] for key in code_to_key.values()}
excel_pids = {pid for pid, _ in rows_by_key}
empty_pids = sorted(excel_pids - matched_pids)

print(f"{len(empty_pids)} pIDs with empty sessions: {empty_pids}")
for pid in empty_pids:
    tags = pid_tags[pid]
    task = "intrusion" if tags.intentionality == 1 else "free_recall"
    n_texts = sum(
        1 for r in rows_by_key.get((pid, task), []) if r.text and r.text.strip()
    )
    print(f"  p{pid} ({task}): {n_texts} texts in Excel — confirms empty")

3 pIDs with empty sessions: [130, 222, 235]
  p130 (intrusion): 0 texts in Excel — confirms empty
  p222 (free_recall): 0 texts in Excel — confirms empty
  p235 (intrusion): 0 texts in Excel — confirms empty


In [30]:
# Validate: does the matched task agree with what intentionality predicts?
task_mismatches = []
for code, (pid, task) in code_to_key.items():
    tags = pid_tags.get(pid)
    if tags is None:
        task_mismatches.append((code, pid, task, "missing PID tags"))
        continue
    expected_task = "intrusion" if tags.intentionality == 1 else "free_recall"
    if task != expected_task:
        task_mismatches.append((code, pid, task, f"expected {expected_task}"))

if task_mismatches:
    print(f"{len(task_mismatches)} task mismatches:")
    for code, pid, task, reason in task_mismatches[:10]:
        print(f"  {code} -> p{pid} matched as {task}, but {reason}")
else:
    print(f"All {len(code_to_key)} task assignments agree with intentionality codes")

All 237 task assignments agree with intentionality codes


## Building output sequences

We build two output representations from the scored documents and Excel event streams:

1. **`recalls`** — the unique-first sequence of clip numbers each participant recalled, the standard input for serial position curves, conditional response probability, and other recall-order analyses. Each entry is a within-list position (1–11). `ScoredSession.unique_clips` already gives us this: presses tagged `[clip_unknown]` or `[invalid_press]` are excluded, and multi-clip presses contribute each clip in tag order with duplicates dropped after first mention.

2. **`recall_items` + `recall_types`** — an interleaved event sequence capturing the full temporal structure of the recall phase. Every film cue, foil scene, recall, and unclassified utterance is recorded as a separate event in temporal order. This matters for two reasons:
   - **Cue attribution**: a foil scene between a cue and a recall breaks the cue→recall link. Without foils in the sequence, downstream analyses would misattribute recalls as cue-elicited when an intervening foil disrupted the association.
   - **Modeling**: CMR-style models can process this via `lax.switch` on event type, dispatching to `reactivate_cue`, `retrieve`, or `process_foil` with type-specific drift rates — requiring only minimal extensions to the existing likelihood code.

The type codes are:

| Code | Event | Value in `recall_items` |
|------|-------|------------------------|
| 0 | Padding | 0 |
| 1 | Film cue | Clip number (1–11) |
| 2 | Recall | Clip number (1–11) |
| 3 | Foil | Scene number (1–68) |
| 4 | Unclassified | 0 |

Type 4 captures button presses where the human rater could not assign a clip (`[clip_unknown]`) or marked the response as invalid (`[invalid_press]`). These are included in the event stream so downstream analyses can decide how to handle them — e.g., treating an unclassified utterance after a film cue as a failed recall attempt that consumes the cue opportunity, rather than silently skipping it and misattributing the cue to a later recall.

We build the interleaved sequence by walking each participant's Excel event stream row by row, emitting an event whenever we encounter a film cue (bg_type=1), foil (bg_type=2), spacebar press with a scored clip (type 2), or spacebar press with an unclassified response (type 4). Black-background rows and digit responses are omitted — this scheme is extensible to additional event types (e.g., black-background events for modeling attentional decay) by adding type codes, not new arrays.

In [31]:
# Build unique-first recall sequences for each matched session.
trial_recalls: dict[tuple[int, str], list[int]] = {}

for code, (pid, task) in code_to_key.items():
    trial_recalls[(pid, task)] = scored_sessions[code].unique_clips

print(f"{len(trial_recalls)} recall sequences\n")

n_recalls = [len(r) for r in trial_recalls.values()]
describe("Unique clips recalled", n_recalls)

237 recall sequences

  Unique clips recalled           mean=8.5  median=9  min=0  max=11


### Validating press alignment

The interleaved sequence consumes scored presses in order as it walks Excel text rows. For this positional alignment to work, the *k*-th scored press must correspond to the *k*-th Excel text entry. But some scored documents contain **empty presses** — the rater noted a spacebar press that has no clip assignment and no text. These have no counterpart in the Excel (which only records rows where the participant typed something), so they shift all subsequent press indices.

We filter these out before alignment: a press is kept if it has non-empty text *or* a clip assignment. After filtering, we check that counts and normalized texts match position-by-position.

In [32]:
# Filter empty presses (no text AND no clip) — these have no Excel counterpart.
def filter_empty_presses(presses: list[ButtonPress]) -> list[ButtonPress]:
    """Keep presses that have text or a clip assignment."""
    return [bp for bp in presses if bp.text.strip() or bp.clip_numbers]


# Check that each matched session has the same number of text entries in both sources.
count_mismatches = []

for code, (pid, task) in code_to_key.items():
    rows = rows_by_key[(pid, task)]
    presses = filter_empty_presses(scored_sessions[code].presses)

    n_excel = sum(1 for r in rows if r.text and r.text.strip())
    n_scored = len(presses)

    if n_excel != n_scored:
        count_mismatches.append((code, pid, task, n_excel, n_scored))

if count_mismatches:
    print(f"{len(count_mismatches)} sessions with count mismatches:")
    for code, pid, task, ne, ns in count_mismatches[:10]:
        print(f"  {code} (p{pid}): {ne} Excel texts, {ns} scored presses")
else:
    print(f"All {len(code_to_key)} sessions have matching text entry counts")

All 237 sessions have matching text entry counts


In [33]:
# Check positional text alignment: k-th scored press should match k-th Excel text.
alignment_issues = []

for code, (pid, task) in code_to_key.items():
    rows = rows_by_key[(pid, task)]
    presses = filter_empty_presses(scored_sessions[code].presses)

    excel_texts = [normalize_text(r.text) for r in rows if r.text and r.text.strip()]
    scored_texts = [normalize_text(bp.text) for bp in presses]

    n_compare = min(len(excel_texts), len(scored_texts))
    if n_compare == 0:
        continue

    mismatches = sum(1 for i in range(n_compare) if excel_texts[i] != scored_texts[i])
    if mismatches > 0:
        alignment_issues.append((code, pid, mismatches, n_compare))

if alignment_issues:
    print(f"{len(alignment_issues)} sessions with positional mismatches:")
    for code, pid, n_mis, n_total in sorted(alignment_issues)[:10]:
        print(f"  {code} (p{pid}): {n_mis}/{n_total} mismatched")
else:
    print(f"All positions match across {len(code_to_key)} sessions")

48 sessions with positional mismatches:
  BGWLU (p107): 1/9 mismatched
  BRIIW (p95): 1/23 mismatched
  CEPRG (p66): 1/69 mismatched
  CHHIO (p112): 1/29 mismatched
  CQUKB (p38): 2/9 mismatched
  CSMTR (p140): 1/18 mismatched
  DGSSB (p113): 1/20 mismatched
  ECLLM (p55): 1/3 mismatched
  FDGIA (p254): 1/16 mismatched
  FIPFF (p51): 1/36 mismatched


### Building the sequences

With empty presses filtered and alignment validated, we walk each session's Excel event stream and emit events into `recall_items` and `recall_types`. The filtered scored presses are consumed in order as we encounter spacebar rows with text.

In [34]:
def build_interleaved(
    rows: list[VrtRow],
    presses: list[ButtonPress] | None = None,
) -> tuple[list[int], list[int]]:
    """Build interleaved event sequence from Excel rows and optional scored presses."""
    items: list[int] = []
    types: list[int] = []
    press_index = 0

    for row in rows:
        # Film cue image -> type 1
        if row.bg_type == 1:
            clip = parse_cue_clip(row.bg_code)
            if clip is not None:
                items.append(clip)
                types.append(1)

        # Foil scene -> type 3
        elif row.bg_type == 2:
            scene = parse_foil_scene(row.bg_code)
            if scene is not None:
                items.append(scene)
                types.append(3)

        # Spacebar press with text -> recall (type 2) or unclassified (type 4)
        if presses is not None and row.text and row.text.strip():
            if press_index < len(presses):
                bp = presses[press_index]
                if bp.clip_numbers:
                    for clip_num in bp.clip_numbers:
                        items.append(clip_num)
                        types.append(2)
                else:
                    # clip_unknown or invalid_press -> unclassified utterance
                    items.append(0)
                    types.append(4)
                press_index += 1

    return items, types

In [35]:
# Build interleaved sequences for all 240 sessions.
trial_events: dict[tuple[int, str], tuple[list[int], list[int]]] = {}

# 237 matched sessions with scored presses (empty presses filtered).
for code, (pid, task) in code_to_key.items():
    rows = rows_by_key[(pid, task)]
    presses = filter_empty_presses(scored_sessions[code].presses)
    trial_events[(pid, task)] = build_interleaved(rows, presses)

# 3 empty sessions — Excel rows still have cues and foils, just no recalls.
for pid in empty_pids:
    task = "intrusion" if pid_tags[pid].intentionality == 1 else "free_recall"
    rows = rows_by_key[(pid, task)]
    trial_events[(pid, task)] = build_interleaved(rows)

print(f"{len(trial_events)} interleaved sequences\n")

# Summary stats by event type.
event_counts = {1: [], 2: [], 3: [], 4: []}
total_counts = []
for key in sorted(trial_events):
    items, types = trial_events[key]
    total_counts.append(len(items))
    for etype in [1, 2, 3, 4]:
        event_counts[etype].append(sum(1 for t in types if t == etype))

describe("Total events per trial", total_counts)
describe("Cue events (type=1)", event_counts[1])
describe("Recall events (type=2)", event_counts[2])
describe("Foil events (type=3)", event_counts[3])
describe("Unclassified events (type=4)", event_counts[4])

240 interleaved sequences

  Total events per trial          mean=107.7  median=106  min=90  max=165
  Cue events (type=1)             mean=22.0  median=22  min=22  max=22
  Recall events (type=2)          mean=16.7  median=15  min=0  max=74
  Foil events (type=3)            mean=68.0  median=68  min=68  max=68
  Unclassified events (type=4)    mean=1.0  median=0  min=0  max=24


## Assembling HDF5 arrays and exporting

We now have all the pieces: recall sequences, interleaved event sequences, and participant metadata. All that remains is to assemble them into zero-padded 2D integer arrays, run safety checks, and save to HDF5.

The final dataset has 11 fields across 240 trials:

| Group | Fields |
|-------|--------|
| Trial-level (6) | subject, task, condition, intervention, intentionality, listLength |
| Presentation (2) | pres_itemnos, pres_itemids |
| Recall-aligned (1) | recalls |
| Interleaved (2) | recall_items, recall_types |

In [36]:
# Determine trial order and array dimensions.
trial_keys = sorted(trial_events.keys())
n_trials = len(trial_keys)
max_recalls = max(len(trial_recalls.get(key, [])) for key in trial_keys)
max_events = max(len(trial_events[key][0]) for key in trial_keys)

print(f"{n_trials} trials, max_recalls={max_recalls}, max_events={max_events}")

# Allocate arrays.
TASK_CODE = {"intrusion": 1, "free_recall": 2}

subject_arr = np.zeros((n_trials, 1), dtype=int)
task_arr = np.zeros((n_trials, 1), dtype=int)
condition_arr = np.zeros((n_trials, 1), dtype=int)
intervention_arr = np.zeros((n_trials, 1), dtype=int)
intentionality_arr = np.zeros((n_trials, 1), dtype=int)
list_length_arr = np.full((n_trials, 1), 11, dtype=int)

pres_itemnos_arr = np.zeros((n_trials, 11), dtype=int)
pres_itemids_arr = np.zeros((n_trials, 11), dtype=int)

recalls_arr = np.zeros((n_trials, max_recalls), dtype=int)
recall_items_arr = np.zeros((n_trials, max_events), dtype=int)
recall_types_arr = np.zeros((n_trials, max_events), dtype=int)

for i, key in enumerate(trial_keys):
    pid, task = key
    tags = pid_tags[pid]

    # Trial-level.
    subject_arr[i, 0] = pid
    task_arr[i, 0] = TASK_CODE[task]
    condition_arr[i, 0] = 1 if tags.emotion == 1 else 2  # 1=emotional, 2=neutral
    intervention_arr[i, 0] = tags.intervention
    intentionality_arr[i, 0] = tags.intentionality

    # Presentation — always 1..11.
    pres_itemnos_arr[i] = np.arange(1, 12)
    if tags.emotion == 1:
        pres_itemids_arr[i] = np.arange(1, 12)   # emotional: 1-11
    else:
        pres_itemids_arr[i] = np.arange(12, 23)  # neutral: 12-22

    # Recalls.
    r = trial_recalls.get(key, [])
    recalls_arr[i, :len(r)] = r

    # Interleaved events.
    items, types = trial_events[key]
    recall_items_arr[i, :len(items)] = items
    recall_types_arr[i, :len(types)] = types

print("Arrays assembled")

240 trials, max_recalls=11, max_events=165
Arrays assembled


In [37]:
# Safety checks and field summary.
dataset = {
    "subject": subject_arr,
    "task": task_arr,
    "condition": condition_arr,
    "intervention": intervention_arr,
    "intentionality": intentionality_arr,
    "listLength": list_length_arr,
    "pres_itemnos": pres_itemnos_arr,
    "pres_itemids": pres_itemids_arr,
    "recalls": recalls_arr,
    "recall_items": recall_items_arr,
    "recall_types": recall_types_arr,
}

for name, arr in dataset.items():
    assert arr.ndim == 2, f"{name} is not 2D: shape={arr.shape}"
    assert arr.dtype in (np.int32, np.int64, int), f"{name} dtype={arr.dtype}"
    assert np.all(arr >= 0), f"{name} has negative values"
    print(f"  {name:20s}  shape={str(arr.shape):>12s}  dtype={arr.dtype}")

print("\nAll checks passed")

  subject               shape=    (240, 1)  dtype=int64
  task                  shape=    (240, 1)  dtype=int64
  condition             shape=    (240, 1)  dtype=int64
  intervention          shape=    (240, 1)  dtype=int64
  intentionality        shape=    (240, 1)  dtype=int64
  listLength            shape=    (240, 1)  dtype=int64
  pres_itemnos          shape=   (240, 11)  dtype=int64
  pres_itemids          shape=   (240, 11)  dtype=int64
  recalls               shape=   (240, 11)  dtype=int64
  recall_items          shape=  (240, 165)  dtype=int64
  recall_types          shape=  (240, 165)  dtype=int64

All checks passed


In [38]:
from jaxcmr.helpers import save_dict_to_hdf5

output_path = ROOT / "data" / "VRT_clips.h5"
save_dict_to_hdf5(dataset, output_path)
print(f"Saved to {output_path}")

Saved to /Users/jordangunn/jaxcmr/data/VRT_clips.h5


### QC summary

In [39]:
print(f"Trials: {n_trials}")
print(f"Subjects: {len(set(subject_arr[:, 0]))}")

print(f"\nTask distribution:")
print(f"  intrusion: {np.sum(task_arr == 1)}")
print(f"  free_recall: {np.sum(task_arr == 2)}")

print(f"\nCondition distribution:")
print(f"  emotional (1): {np.sum(condition_arr == 1)}")
print(f"  neutral (2): {np.sum(condition_arr == 2)}")

print(f"\nRecalls:")
n_per_trial = (recalls_arr != 0).sum(axis=1)
print(f"  mean={n_per_trial.mean():.1f}  max_width={max_recalls}")

print(f"\nInterleaved events:")
n_events_per = (recall_types_arr != 0).sum(axis=1)
print(f"  mean={n_events_per.mean():.1f}  max_width={max_events}")
for etype, name in [(1, "cue"), (2, "recall"), (3, "foil"), (4, "unclassified")]:
    counts = (recall_types_arr == etype).sum(axis=1)
    print(f"  {name:14s}: mean={counts.mean():.1f}  median={np.median(counts):.0f}  min={counts.min()}  max={counts.max()}")

Trials: 240
Subjects: 240

Task distribution:
  intrusion: 120
  free_recall: 120

Condition distribution:
  emotional (1): 120
  neutral (2): 120

Recalls:
  mean=8.4  max_width=11

Interleaved events:
  mean=107.7  max_width=165
  cue           : mean=22.0  median=22  min=22  max=22
  recall        : mean=16.7  median=15  min=0  max=74
  foil          : mean=68.0  median=68  min=68  max=68
  unclassified  : mean=1.0  median=0  min=0  max=24


### Spot-checking against scored documents

Verify that the assembled `recalls` array matches the unique-first clip sequence from the scored documents, and that the interleaved event counts make sense.

In [40]:
spot_codes = sorted(code_to_key.keys())[:5]

for code in spot_codes:
    pid, task = code_to_key[code]
    session = scored_sessions[code]

    # Find trial index.
    trial_idx = trial_keys.index((pid, task))

    # Compare recalls.
    expected = session.unique_clips
    actual = [int(x) for x in recalls_arr[trial_idx] if x != 0]

    status = "OK" if actual == expected else "MISMATCH"
    print(f"{code} (p{pid}, {task}): recalls {status}")
    if actual != expected:
        print(f"  expected: {expected}")
        print(f"  actual:   {actual}")

    # Check interleaved event counts against scored presses.
    n_recall_events = int((recall_types_arr[trial_idx] == 2).sum())
    n_unclassified_events = int((recall_types_arr[trial_idx] == 4).sum())
    n_clip_entries = sum(len(bp.clip_numbers) for bp in session.presses)
    n_unclassified_presses = sum(
        1 for bp in filter_empty_presses(session.presses)
        if not bp.clip_numbers
    )
    recall_match = "OK" if n_recall_events == n_clip_entries else "MISMATCH"
    unclass_match = "OK" if n_unclassified_events == n_unclassified_presses else "MISMATCH"
    print(f"  recall events: {n_recall_events}, scored clips: {n_clip_entries} ({recall_match})")
    print(f"  unclassified events: {n_unclassified_events}, unclassified presses: {n_unclassified_presses} ({unclass_match})")

    # Check cue and foil counts.
    n_cue = int((recall_types_arr[trial_idx] == 1).sum())
    n_foil = int((recall_types_arr[trial_idx] == 3).sum())
    print(f"  cues: {n_cue}, foils: {n_foil}")
    print()

AGMWB (p103, free_recall): recalls OK
  recall events: 9, scored clips: 9 (OK)
  unclassified events: 0, unclassified presses: 0 (OK)
  cues: 22, foils: 68

AJLYW (p219, intrusion): recalls OK
  recall events: 11, scored clips: 11 (OK)
  unclassified events: 0, unclassified presses: 0 (OK)
  cues: 22, foils: 68

ASRNG (p40, free_recall): recalls OK
  recall events: 30, scored clips: 30 (OK)
  unclassified events: 0, unclassified presses: 0 (OK)
  cues: 22, foils: 68

AUQGT (p78, intrusion): recalls OK
  recall events: 2, scored clips: 2 (OK)
  unclassified events: 1, unclassified presses: 1 (OK)
  cues: 22, foils: 68

AXWOO (p257, intrusion): recalls OK
  recall events: 18, scored clips: 18 (OK)
  unclassified events: 1, unclassified presses: 1 (OK)
  cues: 22, foils: 68

